In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [17]:
!pip install dagshub mlflow -q

In [18]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="skoba23",
    repo_name="ML_Assignment_1",
    mlflow=True
)

Initialized MLflow to track repo "skoba23/ML_Assignment_1"

Repository skoba23/ML_Assignment_1 initialized!

In [19]:
model = mlflow.sklearn.load_model("models:/House-Prices-XGBoost/1")
print("Model is saved!")

Model is saved!


In [22]:
df_test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
test_ids = df_test['Id']
df_test = df_test.drop(['Id'], axis=1)

df_test["TotalSF"] = df_test["TotalBsmtSF"] + df_test["1stFlrSF"] + df_test["2ndFlrSF"]
df_test["HouseAge"] = df_test["YrSold"] - df_test["YearBuilt"]
df_test["YearSinceRemodel"] = df_test["YrSold"] - df_test["YearRemodAdd"]

cat_cols = [col for col in df_test.columns if df_test[col].dtype == 'object']
df_test = pd.get_dummies(df_test, columns=cat_cols)

feature_names = model.get_booster().feature_names
df_test = df_test.reindex(columns=feature_names, fill_value=0)

In [ ]:
predictions = model.predict(df_test)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice' : predictions})
submission.to_csv('submission.csv', index = False)
print(submission.head())
print("Submission is saved")